# <center>TIME SERIES | ARIMA | SARIMA MODELS </center>

---------------------------

### What is a Time Series?

A time series is a sequence of observations recorded over time, usually at regular intervals (daily, monthly, quarterly, etc.).

Time series analysis helps us understand structure in the data (trend, seasonality, noise), and forecasting uses that structure to estimate future values.

### Examples

- Monthly product sales
- Daily website traffic
- Hourly power demand
- Quarterly revenue
- Stock prices

### Main Components

- **Trend**: long-term upward or downward movement.
- **Seasonality**: repeating pattern at a fixed interval (e.g., monthly or yearly).

### Modeling and Evaluation

Common models include Naive baseline, Moving Average, Exponential Smoothing, ARIMA, and SARIMA.

Common error metrics include MSE and RMSE.

-------------

### Autocorrelation and Partial Autocorrelation

Before choosing ARIMA/SARIMA terms, we inspect serial dependence in the series.

- **Autocorrelation (ACF)**: correlation between the series and its lagged values.
- **Partial Autocorrelation (PACF)**: correlation at a lag after removing effects of smaller lags.

### Why this matters

Time order matters in forecasting. If lag relationships are ignored, model parameters become unreliable.

### Practical Rule of Thumb

- PACF helps identify the **AR order (p)**.
- ACF helps identify the **MA order (q)**.
- Differencing helps determine **d** by making the series more stationary.

### Note on Durbin-Watson

Durbin-Watson is mainly used for autocorrelation in regression residuals. In this notebook, ACF/PACF and stationarity checks are the primary diagnostics for ARIMA/SARIMA.

---------------

# <center>TIME SERIES MODELING - ARIMA, SARIMA</center>

### ARIMA vs SARIMA

### ARIMA

ARIMA is a univariate forecasting model with three terms:

- **p**: autoregressive lags
- **d**: differencing order
- **q**: moving-average lags

ARIMA is suitable when seasonal effects are absent or already removed.

### SARIMA

SARIMA extends ARIMA by adding seasonal terms:

- **(P, D, Q, m)** where **m** is the seasonal period (for monthly yearly seasonality, `m=12`).

### Workflow used in this notebook

1. Visualize the series.
2. Test stationarity (ADF).
3. Apply differencing where needed.
4. Use ACF/PACF to guide parameter selection.
5. Fit ARIMA and SARIMA.
6. Compare forecast behavior and error metrics.
7. Produce future monthly forecasts.

------------

# <center>PYTHON IMPLEMENTATION</center>

### 1. BASIC STEPS OF A PROJECT:

In [147]:
import warnings

import pandas as pd

warnings.filterwarnings("ignore")
pd.options.plotting.backend = "plotly"

In [148]:
df=pd.read_csv(r'D:\Job\Projects\Analytics\data\raw\time-series\Perrin Freres monthly champagne sales millions.csv')

In [149]:
df.head()

,Month,Perrin Freres monthly champagne sales millions ?64-?72
0,1964-01,2815.0
1,1964-02,2672.0
2,1964-03,2755.0
3,1964-04,2721.0
4,1964-05,2946.0


In [150]:
## Change the Column Names 
df.columns=["Month","Sales"]
df.head()

,Month,Sales
0,1964-01,2815.0
1,1964-02,2672.0
2,1964-03,2755.0
3,1964-04,2721.0
4,1964-05,2946.0


In [151]:
df.tail()

,Month,Sales
102,1972-07,4298.0
103,1972-08,1413.0
104,1972-09,5877.0
105,NaN,NaN
106,Perrin Freres monthly champagne sales millions...,NaN


#### Data quality note

The raw file includes trailing rows with missing sales values. We remove those rows before modeling to keep the time index clean.

In [152]:
## Drop last 2 rows
df.drop(106,axis=0,inplace=True)

In [153]:
df.drop(105,axis=0,inplace=True)

In [154]:
# Convert Month into Datetime
df['Month']=pd.to_datetime(df['Month'])

In [155]:
df.head()

,Month,Sales
0,1964-01-01,2815.0
1,1964-02-01,2672.0
2,1964-03-01,2755.0
3,1964-04-01,2721.0
4,1964-05-01,2946.0


In [156]:
# Set Month as index if needed, ensure datetime index, then enforce monthly-start frequency
df.set_index('Month',inplace=True)

In [157]:
df.head()

,Sales
Month,
1964-01-01,2815.0
1964-02-01,2672.0
1964-03-01,2755.0
1964-04-01,2721.0
1964-05-01,2946.0


In [158]:
df.describe()

,Sales
count,105.000000
mean,4761.152381
std,2553.502601
min,1413.000000
25%,3113.000000
50%,4217.000000
75%,5221.000000
max,13916.000000


--------------

### 2. VISUALIZE THE DATA:

In [162]:
df['Sales'].plot(title="Monthly Champagne Sales").show()

#### Stationarity Check

A stationary series has stable statistical properties over time (mean/variance approximately constant), which is important for ARIMA-family models.

`adfuller` (Augmented Dickey-Fuller test) is used here to test stationarity.

In [122]:
from statsmodels.tsa.stattools import adfuller

In [123]:
test_result=adfuller(df['Sales'])

In [163]:
# H0: non-stationary | H1: stationary
def adfuller_test(sales):
    stat, p, lags, nobs, *_ = adfuller(sales)
    print(f"ADF Test Statistic : {stat}\np-value : {p}\n#Lags Used : {lags}\nNumber of Observations Used : {nobs}")
    print("strong evidence against H0; series is stationary" if p <= 0.05 else "weak evidence against H0; series is non-stationary")

In [164]:
adfuller_test(df['Sales'])

ADF Test Statistic : -1.8335930563276215
p-value : 0.36391577166024586
#Lags Used : 11
Number of Observations Used : 93
weak evidence against H0; series is non-stationary


#### Differencing

Differencing helps reduce trend/seasonal effects and improve stationarity.

In this notebook we use **seasonal differencing with lag 12** because the data is monthly and shows yearly seasonality.

In [165]:
df['Seasonal First Difference']=df['Sales']-df['Sales'].shift(12)

##### Here the value 12 is the number of index values per period of time you are calculating.

In [166]:
df.head(14)

,Sales,Seasonal First Difference
Month,,
1964-01-01,2815.0,NaN
1964-02-01,2672.0,NaN
1964-03-01,2755.0,NaN
1964-04-01,2721.0,NaN
1964-05-01,2946.0,NaN
1964-06-01,3036.0,NaN
1964-07-01,2282.0,NaN
1964-08-01,2212.0,NaN
1964-09-01,2922.0,NaN


In [167]:
## Again test dickey fuller test
adfuller_test(df['Seasonal First Difference'].dropna())

ADF Test Statistic : -7.626619157213166
p-value : 2.0605796968136632e-11
#Lags Used : 0
Number of Observations Used : 92
strong evidence against H0; series is stationary


In [172]:
# After the seasonal first difference, the p-value is less than 0.05, which indicates that we reject the null hypothesis and conclude that the series is stationary.
df['Seasonal First Difference'].plot(title="Seasonal First Difference (Lag 12)").show()

#### After seasonal differencing

The ADF result on the differenced series indicates improved stationarity, so the series is suitable for ARIMA-family modeling.

-----------------

### AR, ACF, and PACF

The following diagnostics help estimate ARIMA terms:

- **PACF** informs likely AR order (`p`).
- **ACF** informs likely MA order (`q`).

These are heuristic guides and should be validated with model fit and forecast error.

### ACF and PACF Diagnostics

In [130]:
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf

In [191]:
from statsmodels.tsa.stattools import acf
pd.Series(acf(df['Sales'].dropna(), nlags=40)).plot.bar(title="Autocorrelation (ACF) - Sales").show()

In [189]:
from statsmodels.tsa.stattools import acf, pacf
data = df['Seasonal First Difference'].dropna()
[pd.Series(func(data, nlags=40)).plot.bar(title=title).show() for func, title in [(pacf, 'PACF'), (acf, 'ACF')]]

### Interpreting ACF/PACF for Model Identification

- For AR models, PACF tends to cut off after lag `p` while ACF decays.
- For MA models, ACF tends to cut off after lag `q` while PACF decays.
- For mixed ARMA behavior, both may decay gradually.

These patterns suggest candidate parameters; final selection should be validated using model diagnostics and error metrics.

#### Practical use in this notebook

- PACF plot is used to choose candidate `p` values.
- ACF plot is used to choose candidate `q` values.
- `d` is chosen based on stationarity checks and differencing.

--------------

### 3. ARIMA Model

ARIMA = **AutoRegressive Integrated Moving Average**

- **AR (p)**: relationship with lagged values
- **I (d)**: differencing order used to improve stationarity
- **MA (q)**: relationship with lagged forecast errors

For seasonal monthly data, ARIMA is often a baseline; SARIMA can improve fit by explicitly modeling seasonality.

In [192]:
# For non-seasonal data
#p=1, d=1, q=0 or 1
from statsmodels.tsa.arima.model import ARIMA

In [194]:
model=ARIMA(df['Sales'],order=(1,1,1))
model_fit=model.fit()

In [195]:
model_fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:                  Sales   No. Observations:                  105
Model:                 ARIMA(1, 1, 1)   Log Likelihood                -952.814
Date:                Fri, 17 Jul 2026   AIC                           1911.627
Time:                        17:35:15   BIC                           1919.560
Sample:                    01-01-1964   HQIC                          1914.841
                         - 09-01-1972                                         
Covariance Type:                  opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.4545      0.114      3.999      0.000       0.232       0.677
ma.L1         -0.9666      0.056    -17.316      0.000      -1.076      -0.857
sigma2      5.226e+06   6.17e+05      8.473      0.000    4.02e+06    6.43e+06
===================================================================================
Ljung-Box (L1) (Q):                   0.91   Jarque-Bera (JB):                 2.59
Prob(Q):                              0.34   Prob(JB):                         0.27
Heteroskedasticity (H):               3.40   Skew:                             0.05
Prob(H) (two-sided):                  0.00   Kurtosis:                         3.77
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [196]:
df['forecast'] = model_fit.predict(start=90, end=103, dynamic=True)
df[['Sales', 'forecast']].plot(title="ARIMA: Actual vs Forecast").show()

### SARIMA Model

SARIMA adds seasonal terms `(P, D, Q, m)` to ARIMA.
For monthly data with yearly pattern, `m = 12` is commonly used.

In [197]:
import statsmodels.api as sm

In [222]:
model=sm.tsa.statespace.SARIMAX(df['Sales'],order=(1, 1, 1),seasonal_order=(1,1,1,12))
results=model.fit()

In [223]:
df['forecast'] = results.predict(start=90, end=103, dynamic=True)
df[['Sales', 'forecast']].plot(title="SARIMA: Actual vs Forecast").show()

##### Plot interpretation

- Blue line: actual sales
- Orange line: model forecast

Closer tracking indicates better in-sample fit. Final model quality should be judged with metrics (RMSE/R2) and residual diagnostics.

In [201]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

actual, predicted = df['Sales'].iloc[90:104], df['forecast'].iloc[90:104]
rmse, r2 = np.sqrt(mean_squared_error(actual, predicted)), r2_score(actual, predicted)
print(f"RMSE: {rmse}\nR2 Score: {r2}\nForecast Error Percentage: {rmse/actual.mean()*100:.2f}%")

RMSE: 464.6986154077518
R2 Score: 0.9733042320457573
Forecast Error Percentage: 8.71%


-------------------

### 4. Forecasting Future Periods

Create future monthly dates, append them to the existing index, and generate model-based forecasts for the next 24 months.

In [224]:
from pandas.tseries.offsets import DateOffset

#Here USING FOR LOOP we are adding some additional data for prediction purpose:

future_dates=[df.index[-1]+ DateOffset(months=x)for x in range(0,24)]

In [225]:
#Convert that list into DATAFRAME:

future_datest_df=pd.DataFrame(index=future_dates[1:],columns=df.columns)

In [205]:
future_datest_df.tail()

,Sales,Seasonal First Difference,forecast
1974-04-01,NaN,NaN,NaN
1974-05-01,NaN,NaN,NaN
1974-06-01,NaN,NaN,NaN
1974-07-01,NaN,NaN,NaN
1974-08-01,NaN,NaN,NaN


In [226]:
#CONCATE THE ORIGINAL AND THE NEWLY CREATED DATASET FOR VISUALIZATION PURPOSE:
future_df=pd.concat([df,future_datest_df])

In [230]:
future_df['forecast'] = results.predict(start=len(df), end=len(df)+23, dynamic=True)
future_df.loc[df.index[-1], 'forecast'] = future_df.loc[df.index[-1], 'Sales']  # connect the two lines, no gap
future_df[['Sales', 'forecast']].apply(pd.to_numeric).plot(title="Sales Forecast", labels={'index': 'Month', 'value': 'Sales'}).show()

## Conclusion

The notebook produces a 24-month sales forecast using ARIMA/SARIMA workflow.
Results are useful for planning, with the understanding that forecasts assume historical patterns continue and may need re-tuning when new data arrives.

------------------------------------------------------------